# Convolutional Neural Networks (CNNs)

## Created by Trung Nghi Tran

# Facial Expression Recognition Using CNNs

# Introduction
Facial Expression Recognition (FER) is an important field of computer vision and artificial intelligence, where the goal is to identify human emotions based on facial expressions. The ability to automatically identify emotions such as happiness, sadness, surprise, anger, etc., has vast applications in areas such as human-computer interaction, healthcare, security, and entertainment.

In this project, we build a model that can recognize basic facial expressions using Convolutional Neural Networks (CNNs). We will train our CNN on a publicly available facial expression dataset to classify images into different emotional categories.

# Why Use CNNs for Facial Expression Recognition?
CNNs are the ideal choice for image classification tasks like FER due to their ability to automatically learn hierarchical features from raw image pixels. Unlike traditional machine learning models, CNNs can detect patterns such as edges, textures, and shapes in images, making them highly effective for tasks involving visual data. CNNs are known to perform well on image recognition tasks because of their capability to capture spatial relationships in images through layers like convolutions, pooling, and fully connected layers.

# Libraries Used

To implement the Facial Expression Recognition model, we will use the following Python libraries:
- PyTorch
- Torchvision
- NumPy
- Matplotlib
- Pandas

## Installation Commands
You can install the required libraries using pip with the following commands:

- pip install torch torchvision
- pip install numpy
- pip install pandas
- pip install matplotlib


# Dataset Source

For this project, we will use the FER2013 dataset, which contains thousands of images labeled with one of the seven basic emotions: happy, sad, surprised, angry, disgusted, fearful, and neutral. The dataset is available from the following source:

FER2013 Dataset: https://www.kaggle.com/datasets/msambare/fer2013

Once you download the dataset, you can access it as CSV files containing the image pixel values and their corresponding labels.

# Implementation

# 1. import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import matplotlib.pyplot as plt

# 2. Load the Dataset (FER-2013)

the data will have 2 folders 1 test and 1 train

In [ ]:
# define the image so that we can use it 
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((48, 48)),  # Resize to 48x48
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize
])

please replace the root to where you puting the files

In [ ]:
# Load dataset 
train_dataset = torchvision.datasets.ImageFolder(root="D:/school/4thSemester/2.AI_Machine_Learning/week10/activities/archive/train", transform=transform)
test_dataset = torchvision.datasets.ImageFolder(root="D:/school/4thSemester/2.AI_Machine_Learning/week10/activities/archive/test", transform=transform)

# DataLoader for batching
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Print class labels
print("Classes:", train_dataset.classes)

at this step we just need to make sure we can load the data and make it suitable to train and yeah we have:

- angry
- disgust
- fear
- happy
- neutral
- sad
- surprise

# 3.  Define CNN Model

In [ ]:
# Define the CNN model architecture
class EmotionCNN(nn.Module):
    def __init__(self):
        super(EmotionCNN, self).__init__() # Initialize the parent class
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1) # 1 input channel (grayscale), 32 output channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # 32 input channels, 64 output channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1) # 64 input channels, 128 output channels
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0) # Max pooling layer
        self.fc1 = nn.Linear(128 * 6 * 6, 256) 
        self.fc2 = nn.Linear(256, len(train_dataset.classes))  # Output neurons = no. of classes
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    # The forward method defines how the input data flows through the network
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Initialize model
model = EmotionCNN()
print(model) # let double check the model structure


the Purpose:

This step sets up a CNN model to process grayscale images, extract meaningful features, and classify them into predefined categories (e.g., emotions). 

-> It prepares the model for training on the dataset.

# 4. Train the Model

I need to create train_losses and train_accurancies array so that we can use it later for the visualization

In [ ]:
# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []
train_accuracies = []

In [ ]:
epochs = 10
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    model.train()  # Set model to training mode
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")


well 72% on the first 10 times run that's pretty not bad

we should make the epoch > 20 so that the accurancy of our model will become better, but it will take so many time so for now I will just keep it at epoch 10 

# 6. Evaluate the Model 

we already see the accurancy number above now let's see some graph about it.

In [ ]:
# Plot the training loss and accuracy
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, marker='o', label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, marker='o', label='Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy')
plt.legend()
plt.grid()

plt.show()

the visualization is match with what we did above to train :)

# 7. test it directly on our webcam

now we gonna use openCV to take the data and then using the model that we just train to processing directly with the video capture by openCV

In [ ]:
# Load OpenCV face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Capture from webcam
cap = cv2.VideoCapture(0)
model.eval()

# Load the trained model to ready 
while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (48, 48))
        face = torch.tensor(face, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0  # Normalize

        with torch.no_grad():
            output = model(face)
            _, predicted = torch.max(output, 1)
            label = train_dataset.classes[predicted.item()]

        # Draw rectangle and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.imshow("Facial Expression Recognition", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'): # Press 'q' to exit
        break

cap.release()
cv2.destroyAllWindows()


because of the accurancy is jsut around 72% so it may not that good

# Conclusion
This project shows how CNNs can be used to recognize facial expressions, achieving good accuracy. While the model works well, it can be improved by trying different CNN designs or using pre-trained models. 

The project main point is highlighting how CNNs are useful for classifying images and can be used in real-world applications like detecting emotions in customer service or interactions between humans and robots.